In [1]:
import os
import numpy as np
from nsd_visuo_semantics.encoding_decoding_analyses.nsd_llm_encoding_model import nsd_llm_encoding_model
from nsd_visuo_semantics.encoding_decoding_analyses.text_to_brain_prediction import text_to_brain_prediction, plot_predicted_brain_contrast
from nsd_visuo_semantics.encoding_decoding_analyses.encoding_decoding_utils import sentences_zoo
from nsd_visuo_semantics.utils.py_plot_brain_utils import pyplot_indiv_subjects, pyplot_subj_avg

EMBEDDING_MODEL_NAME = "all-mpnet-base-v2"
nsd_dir = '/media/chuddy/120876114737F70A/data/NSD'
nsd_derivatives_dir = './results/searchlight/'  # we will put data modified from nsd here
betas_dir = os.path.join(nsd_derivatives_dir, "betas")
base_save_dir = "./results/encoding_decoding_analyses"
encoding_models_dir = f'{base_save_dir}/{EMBEDDING_MODEL_NAME}_encodingModel/fitted_models'
fig_save_dir = f'{base_save_dir}/{EMBEDDING_MODEL_NAME}_encodingModel/figures'
text_to_brain_preds_dir = f'{base_save_dir}/{EMBEDDING_MODEL_NAME}_encodingModel/text_to_brain_preds'

I0000 00:00:1781881913.235813   37445 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781881913.640092   37445 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781881914.849188   37445 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781881914.851544   37445 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will 

In [2]:
# train encoding model for each subject and save model, preds, coeffs
nsd_llm_encoding_model(EMBEDDING_MODEL_NAME, nsd_dir, betas_dir, base_save_dir)

		sub: subj01 fetching condition trials in session: 40subj01 saw 1000 of the 1000
		sub: subj02 fetching condition trials in session: 40subj02 saw 1000 of the 1000
		sub: subj03 fetching condition trials in session: 40subj03 saw 614 of the 1000
		sub: subj04 fetching condition trials in session: 40subj04 saw 515 of the 1000
		sub: subj05 fetching condition trials in session: 40subj05 saw 1000 of the 1000
		sub: subj06 fetching condition trials in session: 40subj06 saw 614 of the 1000
		sub: subj07 fetching condition trials in session: 40subj07 saw 1000 of the 1000
		sub: subj08 fetching condition trials in session: 40subj08 saw 515 of the 1000
		sub: subj01 fetching condition trials in session: 20loading betas for subj01
loading betas for subj01
found some NaN for subj01
Found saved fractional ridge regression, loading...


W0000 00:00:1781881951.010828   37445 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


ValueError: Shapes (515, 327684) and (249, 327674) are incompatible

In [ ]:
# plot performance brain maps
n_subjects, n_vertices = 8, 327684
encoding_preds = np.empty((n_subjects, n_vertices))
for sub in range(8):
    encoding_preds[sub, :] = np.load(f'{encoding_models_dir}/subj{sub+1:02d}_fittedFracridgeEncodingCorrMap.npy')
pyplot_indiv_subjects(encoding_preds, f'{EMBEDDING_MODEL_NAME}_encodingPerf', fig_save_dir, save_type='png')
pyplot_subj_avg(encoding_preds, f'{EMBEDDING_MODEL_NAME}_encodingPerf', fig_save_dir, sig_mask='fdr_bh', save_type='png')

In [ ]:
# predict brain activities from user-derived sentences
for name in ['people', 'places', 'food']:
    text_to_brain_prediction(EMBEDDING_MODEL_NAME, sentences_zoo[name], name, 
                             encoding_models_dir, text_to_brain_preds_dir)

In [ ]:
# plot brain maps for the predicted activities
preds_save_dir = f'{text_to_brain_preds_dir}/cache'
for contrast in [['people', 'places'], ['food', 'people']]:
    plot_predicted_brain_contrast(contrast, preds_save_dir, fig_save_dir, save_type='png', plot_indiv_sub=True, plot_subj_avg=True)